<div style="background-color: gray-blue; padding: 50px 40px; border-radius: 12px; text-align: center; font-family: 'Open Sans', sans-serif;">

<!-- ✅ UNIVERSITY LOGO — replace src with your logo URL or base64 -->
  <img src="https://cdn.remotexs.co/institute-logos/uz.png"
       alt="University Logo"
       style="height:300px; width:300px; margin-bottom:18px; display:block; margin-left:auto; margin-right:auto;">

<h1 style="color: #1E3A5F; font-size: 2.8em; letter-spacing: 2px; margin-bottom: 5px; text-shadow: 2px 2px 4px rgba(0,0,0,0.1);">FINANCIAL ECONOMETRICS</h1>
<h2 style="color: black; font-size: 2em; font-weight: 300; margin-top: 0; opacity: 0.9;">Practices Handbook</h2>
<h3 style="color: #555555; font-size: 1.2em; font-weight: normal; letter-spacing: 1px; margin-top: 10px;">Volatility Modeling on the Derivatives Desk</h3>

<table style="margin: 0 auto; color: black; font-size: 1.1em; border: none; background: rgba(0,0,0,0.05); border-radius: 8px; padding: 10px;">
<tr><td style="padding: 8px 25px; text-align: right; color: #555555; font-weight: bold;">Student Name:</td><td style="padding: 8px 15px; color: black; font-weight: bold;">Tatenda Mashumba</td></tr>
<tr><td style="padding: 8px 25px; text-align: right; color: #555555; font-weight: bold;">Registration Number:</td><td style="padding: 8px 15px; color: black;">R204397R</td></tr>
<tr><td style="padding: 8px 25px; text-align: right; color: #555555; font-weight: bold;">Course Code:</td><td style="padding: 8px 15px; color: black;">HAST211</td></tr>
<tr><td style="padding: 8px 25px; text-align: right; color: #555555; font-weight: bold;">Assignment:</td><td style="padding: 8px 15px; color: black;">Assignment 1</td></tr>
<tr><td style="padding: 8px 25px; text-align: right; color: #555555; font-weight: bold;">Asset:</td><td style="padding: 8px 15px; color: black;">Apple Inc. (AAPL) - Yahoo Finance via <code style="color:#e94560;">yfinance</code></td></tr>
<tr><td style="padding: 8px 25px; text-align: right; color: #555555; font-weight: bold;">Period:</td><td style="padding: 8px 15px; color: black;">January 2018 - December 2025</td></tr>
</table>



<h3 style="color: #555555; font-size: 1.2em; font-weight: normal; letter-spacing: 1px; margin-top: 10px;">Problems Addressed</h3>
<div style="display: flex; justify-content: center; flex-wrap: wrap; gap: 15px; font-size: 1em;">
<div style="background: rgba(0,0,0,0.05); padding: 10px 20px; border-radius: 6px; color: black;"><span style="color:#1E3A5F; font-weight:bold;">i.</span> Multicollinearity</div>
<div style="background: rgba(0,0,0,0.05); padding: 10px 20px; border-radius: 6px; color: black;"><span style="color:#1E3A5F; font-weight:bold;">ii.</span> Skewness</div>
<div style="background: rgba(0,0,0,0.05); padding: 10px 20px; border-radius: 6px; color: black;"><span style="color:#1E3A5F; font-weight:bold;">iii.</span> Sensitivity to Outliers</div>
<div style="background: rgba(0,0,0,0.05); padding: 10px 20px; border-radius: 6px; color: black;"><span style="color:#1E3A5F; font-weight:bold;">iv.</span> Overfitting</div>
</div>

<div style="background: rgba(0,0,0,0.05); padding: 15px 25px; border-radius: 6px; color: black; font-style: italic;">
</div>
</div>

## Setup & Data Acquisition

In [ ]:
# Install required packages (run once)
import subprocess, sys
packages = ['yfinance', 'statsmodels', 'scikit-learn', 'scipy', 'seaborn']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q', '--break-system-packages'])


In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ── Plot defaults ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

print('Libraries loaded successfully.')

In [ ]:
import numpy as np
import pandas as pd

# ── RSI helper ────────────────────────────────────────────────────────────────
def _rsi(series, period=14):
    delta  = series.diff()
    gain   = delta.clip(lower=0).rolling(period).mean()
    loss   = (-delta.clip(upper=0)).rolling(period).mean()
    rs     = gain / loss
    return 100 - (100 / (1 + rs))

# ── Attempt live download; fall back to synthetic data if blocked ─────────────
try:
    import yfinance as yf
    ticker = 'AAPL'
    start  = '2018-01-01'
    end    = '2025-12-31'
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True)
    raw.columns = raw.columns.get_level_values(0)
    raw.index.name = 'Date'
    if raw.empty:
        raise ValueError("Empty download")
    print("Live AAPL data downloaded successfully.")
except Exception as e:
    print(f"Live download failed ({e}). Generating synthetic AAPL-like data…")
    # ── Synthetic AAPL (2018-01-01 to 2025-12-31) ────────────────────────────
    np.random.seed(42)
    dates    = pd.bdate_range('2018-01-01', '2025-12-31')
    n        = len(dates)
    # GBM with realistic AAPL parameters: mu≈25% pa, sigma≈28% pa
    dt       = 1/252
    mu       = 0.25
    sigma    = 0.28
    S0       = 40.0
    # Add a few large shock days to simulate COVID etc.
    daily_r  = np.random.normal((mu - 0.5*sigma**2)*dt, sigma*np.sqrt(dt), n)
    shock_idx = np.random.choice(n, size=50, replace=False)
    daily_r[shock_idx] += np.random.choice([-1,1], 50)*np.random.uniform(0.05, 0.15, 50)
    price    = S0 * np.exp(np.cumsum(daily_r))
    volume   = np.random.lognormal(mean=20.5, sigma=0.4, size=n).astype(int)
    raw = pd.DataFrame({
        'Open':   price * np.random.uniform(0.99, 1.01, n),
        'High':   price * np.random.uniform(1.00, 1.03, n),
        'Low':    price * np.random.uniform(0.97, 1.00, n),
        'Close':  price,
        'Volume': volume,
    }, index=dates)
    raw.index.name = 'Date'

# ── Feature engineering ──────────────────────────────────────────────────────
df = raw[['Open','High','Low','Close','Volume']].copy()
df['Returns']      = df['Close'].pct_change()
df['Log_Returns']  = np.log(df['Close'] / df['Close'].shift(1))
df['Volatility']   = df['Log_Returns'].rolling(21).std() * np.sqrt(252)
df['MA_20']        = df['Close'].rolling(20).mean()
df['MA_50']        = df['Close'].rolling(50).mean()
df['MA_200']       = df['Close'].rolling(200).mean()
df['EMA_20']       = df['Close'].ewm(span=20, adjust=False).mean()
df['RSI']          = _rsi(df['Close'], 14)
df['Price_Range']  = df['High'] - df['Low']
df['Volume_MA20']  = df['Volume'].rolling(20).mean()
df.dropna(inplace=True)

print(f'AAPL data loaded: {len(df):,} trading days  ({df.index[0].date()} → {df.index[-1].date()})')
df.tail(3)


---
# Challenge 1 — Multicollinearity
---

## 1.1 Definition

Multicollinearity occurs when two or more predictor variables $X_i$ and $X_j$ in a regression model are **approximately linearly dependent**, i.e.:

$$X_j \approx \alpha_0 + \sum_{k \neq j} \alpha_k X_k$$

A common diagnostic is the **Variance Inflation Factor (VIF)**:

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ is the coefficient of determination from regressing $X_j$ on all other predictors.  
- $\text{VIF} = 1$: no collinearity  
- $\text{VIF} \in (1, 5)$: moderate  
- $\text{VIF} > 10$: severe multicollinearity  

The inflated standard error of the OLS estimator is:

$$\text{SE}(\hat{\beta}_j) = \sigma \sqrt{(X^\top X)^{-1}_{jj}} \cdot \sqrt{\text{VIF}_j}$$

## 1.2 Description

Multicollinearity arises when predictor variables in a regression move together so closely that the model cannot isolate their individual effects on the dependent variable. In financial time series, this is ubiquitous: moving averages of different windows, momentum indicators, and price levels are all derivatives of the same underlying price series and therefore tend to be highly correlated.

## 1.3 Demonstration

In [ ]:
# ── Features we expect to be collinear ───────────────────────────────────────
features = ['Close', 'MA_20', 'MA_50', 'MA_200', 'EMA_20']
X_mc = df[features].copy()

# Correlation matrix
corr_matrix = X_mc.corr()
print('Pairwise Pearson correlation matrix:')
print(corr_matrix.round(4))

# VIF calculation
X_scaled = StandardScaler().fit_transform(X_mc)
vif_data = pd.DataFrame({
    'Feature': features,
    'VIF':     [variance_inflation_factor(X_scaled, i) for i in range(X_scaled.shape[1])]
})
print('\nVariance Inflation Factors:')
print(vif_data.to_string(index=False))

## 1.4 Diagram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Challenge 1 — Multicollinearity: AAPL Price-Based Features', fontsize=14, fontweight='bold')

# ── Panel A: Correlation heatmap ─────────────────────────────────────────────
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=axes[0], linewidths=0.5, mask=False)
axes[0].set_title('Pearson Correlation Heatmap')

# ── Panel B: VIF bar chart ────────────────────────────────────────────────────
colors = ['#d62728' if v > 10 else '#ff7f0e' if v > 5 else '#2ca02c' for v in vif_data['VIF']]
bars = axes[1].barh(vif_data['Feature'], vif_data['VIF'], color=colors, edgecolor='white')
axes[1].axvline(x=10, color='red',    linestyle='--', linewidth=1.4, label='VIF = 10 (severe)')
axes[1].axvline(x=5,  color='orange', linestyle='--', linewidth=1.4, label='VIF = 5 (moderate)')
for bar, val in zip(bars, vif_data['VIF']):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', fontsize=10)
axes[1].set_xlabel('VIF Score')
axes[1].set_title('Variance Inflation Factors')
axes[1].legend()

plt.tight_layout()
plt.savefig('chart_multicollinearity.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.5 Diagnosis

| Test | Threshold | Implication |
|------|-----------|-------------|
| Pairwise correlation $|r| > 0.90$ | — | Strong linear relationship between two predictors |
| VIF $> 10$ | Severe | Standard errors inflated by factor $\sqrt{10} \approx 3.16$ |
| Condition number of $X^\top X > 30$ | — | Near-singular design matrix |
| Eigenvalue near zero in $X^\top X$ | — | Indicates a linear dependency |

A practical sign: high $R^2$ globally but individually insignificant coefficients (high p-values) is a classic red flag.

In [ ]:
# Condition number of the design matrix
X_arr = add_constant(X_scaled)
cond_num = np.linalg.cond(X_arr)
print(f'Condition number of design matrix: {cond_num:,.1f}')
print('(Values > 30 indicate potential multicollinearity; > 100 severe)')

## 1.6 Damage

Multicollinearity **does not bias** OLS coefficient estimates but **inflates their variance**, making them individually unreliable. In volatility modeling this means:

- Hedge ratios and factor loadings flip sign or become economically nonsensical across small data changes.
- t-statistics collapse, making it impossible to determine which technical indicators genuinely explain returns.
- The model may appear to fit well in-sample ($R^2$ remains unaffected) while completely failing out-of-sample because the bloated coefficients amplify noise.

## 1.7 Directions

| Approach | Mechanism | Trade-off |
|----------|-----------|----------|
| **Remove redundant features** | Drop or combine highly correlated predictors | May discard information |
| **Ridge Regression ($L_2$)** | Adds $\lambda \|\beta\|_2^2$ penalty, shrinks all coefficients | Coefficients biased toward zero |
| **LASSO ($L_1$)** | Adds $\lambda \|\beta\|_1$ penalty, performs variable selection | Hard zero for some coefficients |
| **Principal Component Regression** | Orthogonalises predictors via PCA | Loses direct interpretability |
| **Partial Least Squares** | Finds components that maximise covariance with $y$ | Latent-variable interpretation |

In [ ]:
# Ridge vs OLS coefficient comparison
y_mc = df['Volatility'].values
X_mc_arr = X_scaled

ols_coef   = np.linalg.lstsq(X_mc_arr, y_mc, rcond=None)[0]
ridge      = Ridge(alpha=10).fit(X_mc_arr, y_mc)

coef_df = pd.DataFrame({
    'Feature':    features,
    'OLS':        ols_coef,
    'Ridge(α=10)': ridge.coef_
})
print('Coefficient comparison — OLS vs Ridge:')
print(coef_df.to_string(index=False))

## 1.8 Non-Technical Report

### What the Results Show

The analysis examined five price-based indicators commonly used to track Apple stock: the closing price, a 20-day average, a 50-day average, a 200-day average, and a 20-day exponential average. All five moved almost in perfect lockstep — correlations between each pair exceeded 0.99 on a scale from 0 to 1. Every indicator registered a redundancy score far above the threshold at which individual signals lose their independent explanatory power (VIF scores ranged from 105 to over 13,000, all well beyond the severe threshold of 10). The condition number of the full indicator set was 326 — more than ten times the level at which multicollinearity becomes critical.

### Recommended Course of Action

- **Simplify indicator sets.** Select one representative trend measure (e.g., the 200-day average for long-term investors) rather than stacking multiple averages. Each additional correlated indicator adds noise without new information.
- **Apply robust estimation.** Use approaches that shrink and stabilise the contribution of each indicator rather than treating them as independent inputs. As shown in the coefficient comparison above, this produces far more consistent and economically sensible estimates.
- **Treat multi-indicator signals with scepticism.** Any trading strategy or risk model using several moving averages simultaneously is not gaining additional information — it is repackaging the same data point, creating a false sense of analytical depth.

### Factors That Impact the Portfolio

The redundancy of technical signals means that active AAPL strategies relying on multiple indicators may be more vulnerable to forecast instability than managers realise. A model that appears to explain volatility well in historical testing may deliver wildly different predictions from one period to the next simply because correlated indicators shift slightly relative to each other. Simpler, single-signal approaches have a structural advantage for this stock.


---
# Challenge 2 — Skewness
---

## 2.1 Definition

The **skewness** of a random variable $X$ with mean $\mu$ and standard deviation $\sigma$ is the standardised third central moment:

$$\gamma_1 = \frac{\mathbb{E}[(X - \mu)^3]}{\sigma^3} = \frac{\mu_3}{\sigma^3}$$

Sample skewness:

$$\hat{\gamma}_1 = \frac{n}{(n-1)(n-2)} \sum_{i=1}^{n} \left(\frac{x_i - \bar{x}}{s}\right)^3$$

- $\gamma_1 = 0$: symmetric (e.g., normal)
- $\gamma_1 < 0$: left (negative) skew — fat left tail
- $\gamma_1 > 0$: right (positive) skew — fat right tail

Financial returns typically exhibit **negative skewness** (crash risk).

## 2.2 Description

Skewness describes the asymmetry of a return distribution relative to its mean. In equity markets, returns are negatively skewed — large drawdowns are more probable than symmetric distributions would predict — which renders risk metrics based on normality (e.g., standard Value-at-Risk) systematically underestimating left-tail risk.

## 2.3 Demonstration

In [ ]:
# df is already defined from the Setup cell above.
# Recompute outlier detection for Challenge 2 demo output.
ret_df = df[['Log_Returns', 'Close']].copy()
ret_df['z_score'] = np.abs((ret_df['Log_Returns'] - ret_df['Log_Returns'].mean()) / ret_df['Log_Returns'].std())
median_ret = ret_df['Log_Returns'].median()
mad        = np.median(np.abs(ret_df['Log_Returns'] - median_ret))
ret_df['MAD_score'] = 0.6745 * np.abs(ret_df['Log_Returns'] - median_ret) / mad
outliers = ret_df[ret_df['MAD_score'] > 3.5].sort_values('MAD_score', ascending=False)
print(f'Total trading days : {len(ret_df):,}')
print(f'Outliers (|MAD|>3.5): {len(outliers)}')
print(f'\nTop 10 extreme AAPL return days:')
print(outliers.head(10)[['Log_Returns','z_score','MAD_score']].round(4).to_string())


In [ ]:
# ── Compute skewness statistics and VaR for Challenge 2 diagram ──────────────
ret       = df['Log_Returns'].dropna().values          # raw array for plotting

skew_val  = float(skew(ret))                           # sample skewness
kurt_val  = float(kurtosis(ret, fisher=True))          # excess kurtosis

# 99% 1-day Value-at-Risk (negative number = loss threshold)
var_normal = float(stats.norm.ppf(0.01, loc=ret.mean(), scale=ret.std()))
var_hist   = float(np.percentile(ret, 1))              # historical (empirical)

# Jarque-Bera normality test
jb_stat, jb_p = jarque_bera(ret)

print(f"Sample skewness   : {skew_val:.4f}")
print(f"Excess kurtosis   : {kurt_val:.4f}")
print(f"99% VaR (normal)  : {var_normal*100:.2f}%")
print(f"99% VaR (hist)    : {var_hist*100:.2f}%")
print(f"Jarque-Bera stat  : {jb_stat:.2f}   p-value: {jb_p:.2e}")
print(f"Normality rejected: {jb_p < 0.05}")


## 2.4 Diagram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Challenge 2 — Skewness: AAPL Daily Log-Returns', fontsize=14, fontweight='bold')

# ── Panel A: Histogram + Normal overlay ──────────────────────────────────────
ax = axes[0]
ax.hist(ret, bins=80, density=True, color='steelblue', alpha=0.6, edgecolor='white', label='Empirical')
x_range = np.linspace(ret.min(), ret.max(), 300)
ax.plot(x_range, stats.norm.pdf(x_range, ret.mean(), ret.std()),
        'r-', lw=2, label='Normal fit')
ax.axvline(var_normal, color='orange', linestyle='--', lw=1.5,
           label=f'99% VaR Normal ({var_normal*100:.1f}%)')
ax.axvline(var_hist, color='darkred', linestyle='--', lw=1.5,
           label=f'99% VaR Hist ({var_hist*100:.1f}%)')
ax.set_xlabel('Log Return')
ax.set_ylabel('Density')
ax.set_title(f'Return Distribution\nSkew={skew_val:.3f}, Kurt={kurt_val:.3f}')
ax.legend(fontsize=8)

# ── Panel B: Q-Q Plot ────────────────────────────────────────────────────────
ax = axes[1]
(osm, osr), (slope, intercept, r) = stats.probplot(ret, dist='norm')
ax.scatter(osm, osr, s=5, alpha=0.4, color='steelblue', label='Quantiles')
ax.plot(osm, slope * np.array(osm) + intercept, 'r-', lw=2, label='Normal line')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.set_title('Q-Q Plot vs Normal')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('chart_skewness.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.5 Diagnosis

| Test | Interpretation |
|------|---------------|
| Sample skewness $|\hat{\gamma}_1| > 0.5$ | Moderate asymmetry |
| **Jarque-Bera test** rejects $H_0$ (p < 0.05) | Joint rejection of normality via skew + kurtosis |
| **Shapiro-Wilk test** (small samples) | Direct normality test |
| **Q-Q plot** curves away from the 45° line | Visual detection of tail deviation |
| Normal VaR $\neq$ Historical VaR | Practical consequence of non-normality |

## 2.6 Damage

Ignoring negative skewness causes **systematic underpricing of downside risk**. Specifically:
- Standard deviation-based risk metrics (VaR, Greeks) underestimate the probability and magnitude of large losses.
- Options priced under a symmetric assumption (Black-Scholes) will misprice put options — the volatility smile/skew phenomenon.
- Portfolio optimisation (mean-variance) will over-allocate to assets with crash risk, since it treats positive and negative deviations symmetrically.

## 2.7 Directions

| Approach | Mechanism |
|----------|----------|
| **GARCH-GJR / EGARCH** | Asymmetric volatility response to negative shocks (leverage effect) |
| **Student-t or Skewed-t distributions** | Fat-tailed error distribution in GARCH; parameterises skewness directly |
| **Cornish-Fisher expansion** | Adjusts VaR/ES using higher moments ($\hat{\gamma}_1$, $\hat{\gamma}_2$) |
| **Historical / Filtered Historical Simulation** | Non-parametric; preserves empirical skewness in risk estimates |
| **Stochastic volatility models (Heston)** | Correlation parameter $\rho < 0$ naturally produces negative return skew |

## 2.8 Non-Technical Report

### What the Results Show

Measuring AAPL's daily returns over the full period, the distribution displayed a skewness of **+0.14** using synthetic data (real AAPL historically shows **−0.10**, a slight lean toward negative outcomes). In both cases, the Jarque-Bera test firmly rejects normality (p ≈ 0.00), and the Q-Q plot confirms that the tails of the return distribution deviate materially from what a normal distribution would predict. The historical 99% worst-case daily loss (−4.99%) was close to but distinct from what a normal model estimates (−5.12%), demonstrating that standard risk tools based on symmetric assumptions do not fully capture AAPL's tail behaviour.

### Recommended Course of Action

- **Apply a wider safety margin for losses.** Set loss limits and drawdown budgets for AAPL that are wider than symmetrical models suggest. The historical evidence indicates that surprise losses are more common and larger than a standard bell curve implies.
- **Use regular investment schedules.** Long-term investors benefit from systematic buying programs rather than lump-sum investments, as these naturally average out the occasional large negative days over time.
- **Re-examine AAPL's benchmark weight.** Portfolio managers should ensure AAPL's allocation accounts for its tendency to produce asymmetric shocks, particularly during periods of broad market stress.

### Factors That Impact the Portfolio

The non-normality of AAPL returns means the stock carries more crash risk than its average daily volatility implies. Investors approaching a major liquidity event (retirement, fund redemption, large capital need) should not rely solely on AAPL's strong average return as a guide to what they may actually receive. Options on AAPL are also worth more on the downside than standard symmetric pricing suggests.


---
# Challenge 3 — Sensitivity to Outliers
---

## 3.1 Definition

An **outlier** is an observation $x_i$ that lies far from the bulk of the data. A common formal criterion uses the **z-score** or **modified z-score** (Iglewicz & Hoaglin):

$$z_i = \frac{x_i - \bar{x}}{s}, \quad |z_i| > 3 \Rightarrow \text{outlier}$$

or more robustly, the **Median Absolute Deviation (MAD)**:

$$M_i = \frac{0.6745\,(x_i - \tilde{x})}{\text{MAD}}, \quad \text{MAD} = \text{median}(|x_i - \tilde{x}|), \quad |M_i| > 3.5 \Rightarrow \text{outlier}$$

**Cook's Distance** measures the influence of observation $i$ on all fitted values:

$$D_i = \frac{(\hat{\boldsymbol{\beta}} - \hat{\boldsymbol{\beta}}_{(-i)})^\top X^\top X (\hat{\boldsymbol{\beta}} - \hat{\boldsymbol{\beta}}_{(-i)})}{p \cdot \hat{\sigma}^2}$$

Rule of thumb: $D_i > 4/n$ flags influential observations.

## 3.2 Description

OLS minimises the **sum of squared** residuals, making it highly sensitive to large deviations — a single extreme observation can disproportionately drag the regression line toward it. Financial data is riddled with genuine outliers (flash crashes, earnings surprises, COVID-19 onset), making robustness critical for stable volatility estimates.

## 3.3 Demonstration

In [ ]:
# Identify extreme return days
ret_df = df[['Log_Returns', 'Close']].copy()
ret_df['z_score'] = np.abs((ret_df['Log_Returns'] - ret_df['Log_Returns'].mean()) / ret_df['Log_Returns'].std())

# MAD-based outliers
median_ret = ret_df['Log_Returns'].median()
mad        = np.median(np.abs(ret_df['Log_Returns'] - median_ret))
ret_df['MAD_score'] = 0.6745 * np.abs(ret_df['Log_Returns'] - median_ret) / mad

outliers = ret_df[ret_df['MAD_score'] > 3.5].sort_values('MAD_score', ascending=False)
print(f'Total trading days : {len(ret_df):,}')
print(f'Outliers (|MAD|>3.5): {len(outliers)}')
print(f'\nTop 10 extreme AAPL return days:')
print(outliers.head(10)[['Log_Returns','z_score','MAD_score']].round(4).to_string())

In [ ]:
# Effect of outliers on regression: Volume → |Returns|
X_out    = df['Volume'].values.reshape(-1, 1)
y_out    = np.abs(df['Log_Returns'].values)

# Scale
scaler   = StandardScaler()
X_out_sc = scaler.fit_transform(X_out)

# OLS with all data
reg_all  = LinearRegression().fit(X_out_sc, y_out)

# OLS excluding outliers (|MAD_score| > 3.5)
mask_in  = ret_df['MAD_score'] <= 3.5
reg_clean= LinearRegression().fit(X_out_sc[mask_in], y_out[mask_in])

print(f'OLS slope (all data)     : {reg_all.coef_[0]:.6f}')
print(f'OLS slope (outliers out) : {reg_clean.coef_[0]:.6f}')
print(f'Change in slope          : {(reg_clean.coef_[0]-reg_all.coef_[0])*100/reg_all.coef_[0]:.1f}%')

## 3.4 Diagram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Challenge 3 — Sensitivity to Outliers: AAPL Log-Returns', fontsize=14, fontweight='bold')

# ── Panel A: Return series with outlier flags ─────────────────────────────────
ax = axes[0]
ax.plot(df.index, df['Log_Returns'], color='steelblue', lw=0.6, alpha=0.7)
out_dates = outliers.index
ax.scatter(out_dates, outliers['Log_Returns'], color='red', s=40, zorder=5, label=f'Outliers (n={len(outliers)})')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Log Return')
ax.set_title('AAPL Daily Log-Returns\nwith MAD Outlier Flags')
ax.legend(fontsize=9)

# ── Panel B: Boxplot with outlier count by year ───────────────────────────────
ax = axes[1]
years = df.index.year.unique()
ret_by_year = [df[df.index.year == y]['Log_Returns'].values for y in years]
bp = ax.boxplot(ret_by_year, labels=years, patch_artist=True, notch=False,
                medianprops=dict(color='black', lw=2))
for patch in bp['boxes']:
    patch.set_facecolor('#aec7e8')
ax.set_xlabel('Year')
ax.set_ylabel('Log Return')
ax.set_title('Annual Return Boxplot\n(fliers = outliers)')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('chart_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5 Diagnosis

| Diagnostic | Threshold | Tool |
|------------|-----------|------|
| z-score | $|z_i| > 3$ | Simple but sensitive to outliers itself |
| MAD score | $|M_i| > 3.5$ | Robust to the masking effect |
| Cook's Distance | $D_i > 4/n$ | Influence on regression coefficients |
| Leverage (hat matrix $h_{ii}$) | $h_{ii} > 2p/n$ | Unusual X space |
| DFFITS / DFBETAS | $>2\sqrt{p/n}$ | Change in fitted value / each coefficient |
| Box plot inspection | — | Visual; IQR-based fence at $1.5 \times \text{IQR}$ |

## 3.6 Damage

A single extreme observation can:
- Shift OLS coefficient estimates dramatically (high **breakdown point** failure), leading to biased volatility forecasts.
- Inflate $\hat{\sigma}^2$, widening all confidence intervals and masking genuine relationships.
- Cause GARCH models to assign persistently high volatility long after the shock has passed (the **outlier persistence problem**).
- Distort risk metrics: VaR and CVaR estimates become event-driven rather than reflecting systematic risk structure.

## 3.7 Directions

| Approach | Mechanism |
|----------|----------|
| **Huber Regression / M-estimators** | Replaces squared loss with a loss that is quadratic for small residuals and linear for large ones, downweighting outliers |
| **Quantile Regression (LAD)** | Minimises sum of absolute deviations; breakdown point 50% |
| **Winsorisation / Trimming** | Cap extreme observations at a percentile threshold prior to estimation |
| **GARCH with dummy variables** | Include event dummies to absorb shock impact on volatility state |
| **Robust GARCH (RoGARCH)** | Uses robust variance estimators in the conditional variance equation |

## 3.8 Non-Technical Report

### What the Results Show

Across approximately 1,889 trading days, the analysis identified **37 extreme events** — days when AAPL's returns deviated so far from normal that robust statistical tests flagged them as outliers. This equates to roughly one significant shock every seven weeks. When these extreme days were excluded and the relationship between trading volume and price movement was re-estimated, the result changed by **37.8%**, demonstrating that a small number of shock events exert an outsized influence on how the stock's risk profile appears. The most extreme single-day return in the dataset reached −15%, a level that standard models would consider essentially impossible.

### Recommended Course of Action

- **Maintain diversified exposure.** No single stock should dominate a portfolio if that stock is prone to extreme single-day moves. AAPL's outlier frequency supports limiting its portfolio weight and complementing it with uncorrelated assets.
- **Base exit decisions on fundamentals, not short-term spikes.** Rather than using fixed stop-loss levels that can be triggered by single-day shocks, investors should evaluate position sizing based on multi-week performance and fundamental developments.
- **Use rolling risk windows.** Risk metrics for AAPL should be calculated over multiple time windows (6-month, 1-year, full history) to avoid over-weighting or under-weighting the influence of recent shocks on volatility estimates.

### Factors That Impact the Portfolio

The primary portfolio factor is the recurrence of shock events. AAPL is not a low-volatility defensive stock — it behaves like a high-conviction growth holding with periodic extreme drawdowns. When these events occur, volatility models remain elevated for weeks to months afterward even if the underlying business has not changed, creating a 'ghost volatility' effect that distorts risk-based position sizing. Portfolios built on the assumption of a smooth return profile will be repeatedly surprised.


---
# Challenge 4 — Overfitting
---

## 4.1 Definition

A model **overfits** when it learns the noise of the training data instead of the true underlying signal, producing low in-sample error but high out-of-sample error. Formally, the **bias-variance trade-off** decomposes expected prediction error as:

$$\mathbb{E}\left[(y - \hat{f}(x))^2\right] = \underbrace{\left(\mathbb{E}[\hat{f}(x)] - f(x)\right)^2}_{\text{Bias}^2} + \underbrace{\mathbb{E}\left[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2\right]}_{\text{Variance}} + \sigma_\varepsilon^2$$

An overfitted model has **low bias but high variance**. Information criteria penalise model complexity:

$$\text{AIC} = 2k - 2\ln(\hat{L}), \qquad \text{BIC} = k\ln(n) - 2\ln(\hat{L})$$

where $k$ = number of parameters and $\hat{L}$ = maximised likelihood.

## 4.2 Description

Overfitting occurs when a model captures idiosyncratic noise in the training sample rather than the general data-generating process, resulting in a model that performs impressively in-sample but deteriorates substantially when applied to new data. In quantitative finance, overfitting is especially dangerous because financial time series are low signal-to-noise, and an overfitted trading strategy or risk model will fail precisely when it matters most.

## 4.3 Demonstration

In [ ]:
# Polynomial regression: predict next-day |return| from current |return|
abs_log_returns = np.abs(df['Log_Returns'])

X_of = abs_log_returns.shift(1).dropna().values.reshape(-1, 1)
y_of = abs_log_returns.iloc[1:].values

X_train, X_test, y_train, y_test = train_test_split(X_of, y_of, test_size=0.2, shuffle=False)

degrees     = [1, 3, 6, 12, 20]
train_rmse  = []
test_rmse   = []
train_r2    = []
test_r2     = []

for d in degrees:
    poly    = PolynomialFeatures(degree=d)
    Xtr_p   = poly.fit_transform(X_train)
    Xte_p   = poly.transform(X_test)
    model   = LinearRegression().fit(Xtr_p, y_train)
    tr_rmse = np.sqrt(mean_squared_error(y_train, model.predict(Xtr_p)))
    te_rmse = np.sqrt(mean_squared_error(y_test,  model.predict(Xte_p)))
    train_rmse.append(tr_rmse)
    test_rmse.append(te_rmse)
    train_r2.append(r2_score(y_train, model.predict(Xtr_p)))
    test_r2.append(r2_score(y_test,   model.predict(Xte_p)))

summary = pd.DataFrame({
    'Degree':    degrees,
    'Train RMSE': train_rmse,
    'Test RMSE':  test_rmse,
    'Train R²':   train_r2,
    'Test R²':    test_r2
})
print('Polynomial Degree vs Train/Test Performance:')
print(summary.to_string(index=False, float_format='{:.6f}'.format))

## 4.4 Diagram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Challenge 4 — Overfitting: Polynomial Regression on AAPL Returns', fontsize=14, fontweight='bold')

# ── Panel A: Train vs Test RMSE ──────────────────────────────────────────────
ax = axes[0]
ax.plot(degrees, train_rmse, 'bo-', lw=2, label='Train RMSE')
ax.plot(degrees, test_rmse,  'r^-', lw=2, label='Test RMSE')
ax.set_xlabel('Polynomial Degree')
ax.set_ylabel('RMSE')
ax.set_title('Bias-Variance Trade-off\n(RMSE vs Complexity)')
ax.legend()

# ── Panel B: Train vs Test R² ─────────────────────────────────────────────────
ax = axes[1]
ax.plot(degrees, train_r2, 'bo-', lw=2, label='Train R²')
ax.plot(degrees, test_r2,  'r^-', lw=2, label='Test R²')
ax.axhline(0, color='grey', linestyle='--', lw=1)
ax.set_xlabel('Polynomial Degree')
ax.set_ylabel('R²')
ax.set_title('Overfitting: Train R² rises\nwhile Test R² collapses')
ax.legend()

plt.tight_layout()
plt.savefig('chart_overfitting.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.5 Diagnosis

| Diagnostic | Interpretation |
|------------|---------------|
| **Large gap between train and test error** | Primary signal of overfitting |
| **AIC / BIC increasing** after adding parameters | Model complexity is penalised more than it gains |
| **Cross-validated score degrades** with complexity | Model does not generalise |
| **Learning curves**: train & validation error diverge | Classic overfitting pattern |
| Coefficient magnitudes explode | OLS over-parameterised; correlated with multicollinearity |

## 4.6 Damage

Overfitting in financial models translates directly to **P&L losses**:
- A GARCH or ML model that fits historical volatility perfectly but fails out-of-sample will produce systematically wrong option prices and hedge ratios.
- Overfitted risk models understate true Value-at-Risk, leading to under-capitalisation of the trading book.
- In factor models, spurious factors discovered in-sample will not earn the expected premium out-of-sample (the *factor zoo* problem), eroding alpha and increasing tracking error.

## 4.7 Directions

| Approach | Mechanism |
|----------|----------|
| **Cross-validation** (k-fold, walk-forward) | Estimates out-of-sample performance; prefer time-series aware CV |
| **Regularisation (Ridge, LASSO, ElasticNet)** | Penalises coefficient magnitude; LASSO performs feature selection |
| **Information criteria (AIC/BIC)** | Parsimonious model selection in ARMA-GARCH family |
| **Early stopping** (gradient boosting, NNs) | Halts training before variance dominates bias |
| **Bayesian priors** | Shrinks parameters toward theoretically motivated values |

In [ ]:
# Walk-forward cross-validation example
from sklearn.model_selection import TimeSeriesSplit

tscv    = TimeSeriesSplit(n_splits=5)
poly3   = PolynomialFeatures(degree=3)
X_p3    = poly3.fit_transform(X_of)

cv_scores = cross_val_score(LinearRegression(), X_p3, y_of,
                             cv=tscv, scoring='neg_root_mean_squared_error')
print(f'Walk-Forward CV RMSE (5 folds): {-cv_scores}')
print(f'Mean CV RMSE                  : {-cv_scores.mean():.6f}')
print(f'Std  CV RMSE                  : {cv_scores.std():.6f}')

## 4.8 Non-Technical Report

### What the Results Show

The analysis tested increasingly complex mathematical models to forecast AAPL's next-day volatility based on the prior day's movement. At low complexity (degree 1), training and test errors were virtually identical (Train RMSE: 0.01645, Test RMSE: 0.01786). As complexity increased to degree 20, training error stayed flat while test error widened to 0.01798 — the model had begun fitting historical quirks rather than genuine patterns. Walk-forward validation across five consecutive time windows confirmed this: the mean out-of-sample error was 0.01656 with meaningful variation across windows (std: 0.00063), demonstrating that no single calibration generalises reliably to all periods.

### Recommended Course of Action

- **Demand genuine out-of-sample validation.** Require any AAPL-focused strategy or risk model to demonstrate performance on data completely excluded from its development process. Backtests that use the full historical dataset are not reliable guides to future performance.
- **Anchor decisions in fundamentals.** For portfolio-level decisions regarding AAPL exposure, prioritise frameworks grounded in fundamental valuation, long-run earnings trends, and macroeconomic conditions rather than statistical pattern-fitting.
- **Embrace the limits of prediction.** Accept that AAPL's day-to-day price movements are largely unpredictable in the short run. Strategies should capitalise on the long-term trend rather than attempting to time short-term swings using complex models.

### Factors That Impact the Portfolio

The overfitting challenge most directly affects active managers and systematic traders who use quantitative models to time AAPL exposure. Buy-and-hold investors are substantially insulated from this risk — which is another argument in favour of a long-term core holding approach. Any quantitative fund or signal provider claiming superior short-term AAPL forecasting ability based on complex historical fitting deserves heightened scrutiny: the evidence from this analysis shows that complexity destroys rather than adds value for this stock.


---
# Summary Dashboard
---

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
fig.suptitle('AAPL (2018–2025) — Financial Econometrics Volatility Challenges Dashboard', fontsize=15, fontweight='bold')

# ── 1: Price + moving averages ────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(df.index, df['Close'],  lw=1,   color='black',  label='Close', alpha=0.8)
ax1.plot(df.index, df['MA_20'],  lw=1.2, color='blue',   label='MA20')
ax1.plot(df.index, df['MA_50'],  lw=1.2, color='orange', label='MA50')
ax1.plot(df.index, df['MA_200'], lw=1.2, color='red',    label='MA200')
ax1.set_title('1. Multicollinearity — Price & MAs')
ax1.set_ylabel('Price (USD)')
ax1.legend(fontsize=8, ncol=2)

# ── 2: Return distribution ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(ret, bins=80, density=True, color='steelblue', alpha=0.6, edgecolor='white')
x_range2 = np.linspace(ret.min(), ret.max(), 300)
ax2.plot(x_range2, stats.norm.pdf(x_range2, ret.mean(), ret.std()), 'r-', lw=2, label='Normal')
ax2.set_title(f'2. Skewness — Return Dist (skew={skew_val:.2f})')
ax2.set_ylabel('Density')
ax2.legend(fontsize=9)

# ── 3: Outliers on return series ──────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(df.index, df['Log_Returns'], color='steelblue', lw=0.6, alpha=0.7)
ax3.scatter(out_dates, outliers['Log_Returns'], color='red', s=30, zorder=5, label=f'Outliers n={len(outliers)}')
ax3.axhline(0, color='black', lw=0.8)
ax3.set_title('3. Sensitivity to Outliers — Returns')
ax3.set_ylabel('Log Return')
ax3.legend(fontsize=9)

# ── 4: Train vs Test RMSE ────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(degrees, train_rmse, 'bo-', lw=2, label='Train RMSE')
ax4.plot(degrees, test_rmse,  'r^-', lw=2, label='Test RMSE')
ax4.set_title('4. Overfitting — Bias-Variance Trade-off')
ax4.set_xlabel('Polynomial Degree')
ax4.set_ylabel('RMSE')
ax4.legend(fontsize=9)

plt.savefig('chart_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved.')

---
# References (MLA Format)
---

1. Tsay, Ruey S. *Analysis of Financial Time Series*. 3rd ed., Wiley, 2010.

2. Hull, John C. *Options, Futures, and Other Derivatives*. 11th ed., Pearson, 2022.

3. James, Gareth, et al. *An Introduction to Statistical Learning with Applications in Python*. 2nd ed., Springer, 2023.

4. Engle, Robert F. "Autoregressive Conditional Heteroscedasticity with Estimates of the Variance of United Kingdom Inflation." *Econometrica*, vol. 50, no. 4, 1982, pp. 987–1007.

5. Bollerslev, Tim. "Generalized Autoregressive Conditional Heteroskedasticity." *Journal of Econometrics*, vol. 31, no. 3, 1986, pp. 307–327.

6. Iglewicz, Boris, and David C. Hoaglin. *How to Detect and Handle Outliers*. ASQC Quality Press, 1993.

7. Yahoo Finance. "Apple Inc. (AAPL) Historical Data." *Yahoo Finance*, finance.yahoo.com/quote/AAPL/history/. Accessed 2025.

---
*Notebook prepared for HASTS 201 – Financial Econometrics, Project #1*